In [3]:
from skyfield import api, almanac
from datetime import datetime, timezone

# Load ephemeris
ts = api.load.timescale()
eph = api.load('de421.bsp')

def get_season_events(year):
    t0 = ts.utc(year, 1, 1)
    t1 = ts.utc(year, 12, 31)
    times, events = almanac.find_discrete(t0, t1, almanac.seasons(eph))
    season_dict = {}
    for ti, ev in zip(times, events):
        dt = ti.utc_datetime()
        # Ensure UTC-aware
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        if ev == 0:
            season_dict['mar_equinox'] = dt
        elif ev == 1:
            season_dict['jun_solstice'] = dt
        elif ev == 2:
            season_dict['sep_equinox'] = dt
        elif ev == 3:
            season_dict['dec_solstice'] = dt
    return season_dict

def earth_orbit_angle(dt):
    # If naive datetime, treat as UTC
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    
    year = dt.year
    seasons = get_season_events(year)
    
    season_angles = [
        (seasons['dec_solstice'], 0),
        (seasons['mar_equinox'], 90),
        (seasons['jun_solstice'], 180),
        (seasons['sep_equinox'], 270),
        (get_season_events(year + 1)['dec_solstice'], 360)
    ]
    
    for i in range(len(season_angles) - 1):
        t0, a0 = season_angles[i]
        t1, a1 = season_angles[i+1]
        if t0 <= dt < t1:
            fraction = (dt - t0).total_seconds() / (t1 - t0).total_seconds()
            angle = a0 + fraction * (a1 - a0)
            return angle
    return None

# Examples
angle1 = earth_orbit_angle(datetime(2025, 11, 20, 12))  # naive treated as UTC
angle2 = earth_orbit_angle(datetime(2025, 11, 20, 12, tzinfo=timezone.utc))  # aware datetime
angle_now = earth_orbit_angle(datetime.now(timezone.utc))
print(f"Earth orbital angle 1: {angle1:.2f}°")
print(f"Earth orbital angle 2: {angle2:.2f}°")
print(f"Earth orbital angle_now: {angle_now:.2f}°")

Earth orbital angle 1: 281.62°
Earth orbital angle 2: 281.62°
Earth orbital angle_now: 281.64°


In [16]:


# FuXi list of 88 variables
fuxi_variables = [

    # ------------------------------
    # Upper-air variables (Z, T, U, V, Q)
    # ------------------------------

    # z
    "z_50","z_100","z_150","z_200","z_250",
    "z_300","z_400","z_500","z_600","z_700",
    "z_850","z_925","z_1000",

    # t
    "t_50","t_100","t_150","t_200","t_250",
    "t_300","t_400","t_500","t_600","t_700",
    "t_850","t_925","t_1000",

    # u
    "u_50","u_100","u_150","u_200","u_250",
    "u_300","u_400","u_500","u_600","u_700",
    "u_850","u_925","u_1000",

    # v
    "v_50","v_100","v_150","v_200","v_250",
    "v_300","v_400","v_500","v_600","v_700",
    "v_850","v_925","v_1000",

    # q
    "q_50","q_100","q_150","q_200","q_250",
    "q_300","q_400","q_500","q_600","q_700",
    "q_850","q_925","q_1000",

    # ------------------------------
    # Surface variables
    # ------------------------------

    # temp/humidity
    "t2m","d2m","sst",

    # winds
    "u10","v10","u100","v100",

    # pressure
    "msl",

    # clouds
    "lcc","mcc","hcc","tcc",

    # radiation
    "ssr","ssrd","fdir","ttr","tp",

    # wave / storm variables
    "mdts","mdww","mpts","mpww","shts","shww",

    # ------------------------------
    # Slots for your 13 extra variables
    # ------------------------------

    None, None, None, None, None,
    None, None, None, None, None,
    None, None, None
]

len([v for v in fuxi_variables if v is not None])

88